# Bank Loan Risk & Customer Behavior Analysis

In [2]:
# Install Liaberies

import pandas as pd
import numpy as np

## File Import

In [4]:
customers = pd.read_csv(r"C:\Users\goswa\OneDrive\Documents\MY PROJECTS\END TO END\Banking\Datasets/Customers.csv")
loans = pd.read_csv(r"C:\Users\goswa\OneDrive\Documents\MY PROJECTS\END TO END\Banking\Datasets\Loans.csv")
transactions = pd.read_csv(r"C:\Users\goswa\OneDrive\Documents\MY PROJECTS\END TO END\Banking\Datasets\Transactions.csv")

## Inspection Of Customers Data

In [6]:
print(customers.head(10))

  Customer_ID  Age   Gender  Income Employment_Type Years_of_Employment  \
0   CUST04290   55   Female   69347            Sal.                  23   
1   CUST05421   54     MALE     NaN       Full-Time                  32   
2   CUST01159   38        f   69445      Freelancer                   7   
3   CUST05788   57        F  104759            Sal.                   9   
4   CUST01395   62        F     NaN        SALARIED                   2   
5   CUST02960   47   FEMALE  133789  Business owner                  20   
6   CUST02458  NaN     male   20314   Self Employed                  30   
7   CUST01070   47     MALE   24405   Self Employed                  20   
8   CUST01874   54      NaN   53822   Self-Employed                  16   
9   CUST02148   38   Female   40155              SE                   4   

  Credit_Score  
0          715  
1          772  
2          666  
3          763  
4          608  
5          784  
6          NaN  
7          741  
8          850  
9   

In [7]:
print(customers.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6090 entries, 0 to 6089
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Customer_ID          6090 non-null   object
 1   Age                  5900 non-null   object
 2   Gender               5866 non-null   object
 3   Income               5820 non-null   object
 4   Employment_Type      5874 non-null   object
 5   Years_of_Employment  5762 non-null   object
 6   Credit_Score         5836 non-null   object
dtypes: object(7)
memory usage: 333.2+ KB
None


In [8]:
customers.shape

(6090, 7)

In [9]:
# Handling Duplicates

customers.duplicated().sum()

90

In [11]:
# Delete Duplicates from customers
customers.drop_duplicates(inplace=True)

In [18]:
# Fixing Datatypes
customers['Age'] = pd.to_numeric(customers['Age'], errors='coerce')
customers['Income'] = pd.to_numeric(customers['Income'], errors='coerce')
customers['Years_of_Employment'] = pd.to_numeric(customers['Years_of_Employment'], errors='coerce')
customers['Credit_Score'] = pd.to_numeric(customers['Credit_Score'], errors='coerce')

In [20]:
# Removing negetive years
customers.loc[customers['Years_of_Employment'] < 0, 'Years_of_Employment'] = 0

In [22]:
# Fixing Outliers

# Income
Q1 = customers['Income'].quantile(0.25)
Q3 = customers['Income'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

customers['Income'] = customers['Income'].clip(lower, upper)



# Years_of_Employment
Q1 = customers['Years_of_Employment'].quantile(0.25)
Q3 = customers['Years_of_Employment'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

customers['Years_of_Employment'] = customers['Years_of_Employment'].clip(lower, upper)


# Credit_Score
Q1 = customers['Credit_Score'].quantile(0.25)
Q3 = customers['Credit_Score'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

customers['Credit_Score'] = customers['Credit_Score'].clip(lower, upper)

In [24]:
customers.describe()

,Age,Income,Years_of_Employment,Credit_Score
count,5751.000000,5312.000000,5553.000000,5653.000000
mean,43.883499,87738.901191,11.069692,708.536175
std,20.302726,52543.467130,9.737088,88.744289
min,-5.000000,-5000.000000,0.000000,466.000000
25%,31.000000,49751.250000,3.000000,649.000000
50%,43.000000,80174.500000,8.000000,712.000000
75%,55.000000,116750.500000,17.000000,771.000000
max,200.000000,217249.375000,38.000000,954.000000


In [26]:
customers = customers[
    (customers['Age'] >= 18) &
    (customers['Age'] <= 80)
]

In [28]:
customers = customers[
    customers['Income'] >= 0
]

In [30]:
customers['Employment_Type'].value_counts()

Employment_Type
Salaried          506
SALARIED          473
salaried          461
Full-Time         443
Sal.              439
Self Employed     205
self-employed     202
Self-Employed     193
SE                169
Self employed     162
Entrepreneur      140
Business Owner    133
BUSINESS OWNER    133
Business owner    131
Biz Owner         126
Freelancer        108
Free Lancer       104
freelancer        102
Gig Worker         96
FREELANCER         81
unemployed         58
Unemployed         51
UNEMPLOYED         38
Not Employed       35
Jobless            34
                   31
Unknown            29
Name: count, dtype: int64

In [32]:
# Null Checking

print(customers.isnull().sum())

Customer_ID              0
Age                      0
Gender                 185
Income                   0
Employment_Type        183
Years_of_Employment    369
Credit_Score           269
dtype: int64


In [34]:
# Handling Null Values

customers['Age'] = customers['Age'].fillna(customers['Age'].median())
customers['Gender'] = customers['Gender'].fillna(customers['Gender'].mode()[0])
customers['Income'] = customers['Income'].fillna(customers['Income'].mean())
customers['Employment_Type'] = customers['Employment_Type'].fillna(customers['Employment_Type'].mode()[0])
customers['Years_of_Employment'] = customers['Years_of_Employment'].fillna(customers['Years_of_Employment'].mean())
customers['Credit_Score'] = customers['Credit_Score'].fillna(customers['Credit_Score'].mean())

In [36]:
customers['Gender'].value_counts(dropna=False)

Gender
M             591
m             402
F             383
female        379
male          378
FEMALE        375
 Female       372
Male          372
Male          368
MALE          362
f             362
Female        361
Unknown        36
               36
Non-Binary     22
NB             21
non-binary     18
Non binary     16
Other          12
Name: count, dtype: int64

In [38]:
# Fixing Inconsistent Formats

customers['Gender'] = customers['Gender'].str.lower().str.strip()

gender_map = {
    'm': 'male', 'male': 'male',
    'f': 'female', 'female': 'female',
    'unknown': 'unknown',
    'nb': 'non-binary',
    'non-binary': 'non-binary',
    'non binary': 'non-binary',
    'other': 'other'
}

customers['Gender'] = customers['Gender'].map(gender_map)


# filling null Values

customers['Gender'] = customers['Gender'].fillna('Unknown')

In [40]:
customers['Employment_Type'].value_counts(dropna=False)

Employment_Type
Salaried          689
SALARIED          473
salaried          461
Full-Time         443
Sal.              439
Self Employed     205
self-employed     202
Self-Employed     193
SE                169
Self employed     162
Entrepreneur      140
Business Owner    133
BUSINESS OWNER    133
Business owner    131
Biz Owner         126
Freelancer        108
Free Lancer       104
freelancer        102
Gig Worker         96
FREELANCER         81
unemployed         58
Unemployed         51
UNEMPLOYED         38
Not Employed       35
Jobless            34
                   31
Unknown            29
Name: count, dtype: int64

In [42]:
# Handling Missing / Unmapped Values

customers['Employment_Type'] = customers['Employment_Type'].str.lower().str.strip()

In [44]:
# Fixing Inconsistent Formats

emp_map = {
    'salaried': 'salaried',
    'full-time': 'salaried',
    'sal.': 'salaried',

    'self employed': 'self-employed',
    'self-employed': 'self-employed',
    'se': 'self-employed',

    'entrepreneur': 'business_owner',
    'business owner': 'business_owner',
    'biz owner': 'business_owner',

    'freelancer': 'freelancer',
    'free lancer': 'freelancer',
    'gig worker': 'freelancer',

    'unemployed': 'unemployed',
    'jobless': 'unemployed',
    'not employed': 'unemployed'
}

customers['Employment_Type'] = customers['Employment_Type'].map(emp_map)

In [46]:
# Handling Missing / Unmapped Values

customers['Employment_Type'] = customers['Employment_Type'].fillna('Unknown')

# Inspection Of Loans Data

In [49]:
print(loans.head(10))

      Loan_ID Customer_ID Loan_Amount       Loan_Type Interest_Rate Loan_Term  \
0  LOAN006290   CUST01050     4255207       home loan          7.44        24   
1  LOAN005834   CUST01901     1762680    Housing Loan          6.95       180   
2  LOAN006703   CUST02643     1808421       Home Loan           9.2        12   
3  LOAN010147   CUST00709           0        Mortgage          7.93        24   
4  LOAN006261   CUST05735      234423   Personal Loan         11.83       240   
5  LOAN003394   CUST00843     1428684   Business Loan         12.53        36   
6  LOAN001778   CUST01500      296863  EDUCATION LOAN          7.68        12   
7  LOAN004943   CUST01140      473199              PL         12.24       240   
8  LOAN003405   CUST03824      410177  education loan         10.25        60   
9  LOAN002826   CUST04827     4854307       HOME LOAN           9.6       NaN   

  Loan_Status Debt_to_Income_Ratio  
0     default                  NaN  
1  Fully Paid                  0.0

In [51]:
print(loans.shape)

(12180, 8)


In [53]:
print(loans.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12180 entries, 0 to 12179
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Loan_ID               12180 non-null  object
 1   Customer_ID           11986 non-null  object
 2   Loan_Amount           11754 non-null  object
 3   Loan_Type             11829 non-null  object
 4   Interest_Rate         11743 non-null  object
 5   Loan_Term             11823 non-null  object
 6   Loan_Status           11820 non-null  object
 7   Debt_to_Income_Ratio  11638 non-null  object
dtypes: object(8)
memory usage: 761.4+ KB
None


In [55]:
# Fixing Dtypes

loans['Loan_Amount'] = pd.to_numeric(loans['Loan_Amount'], errors='coerce')
loans['Interest_Rate'] = pd.to_numeric(loans['Interest_Rate'], errors='coerce')
loans['Loan_Term'] = pd.to_numeric(loans['Loan_Term'], errors='coerce')
loans['Debt_to_Income_Ratio'] = pd.to_numeric(loans['Debt_to_Income_Ratio'], errors='coerce')

print(loans.dtypes)

Loan_ID                  object
Customer_ID              object
Loan_Amount             float64
Loan_Type                object
Interest_Rate           float64
Loan_Term               float64
Loan_Status              object
Debt_to_Income_Ratio    float64
dtype: object


In [57]:
loans['Loan_Type'].value_counts(dropna=False)

Loan_Type
Personal          714
PL                702
PERSONAL LOAN     695
personal loan     683
Personal Loan     676
Mortgage          624
HOME LOAN         597
Housing Loan      595
Home Loan         592
home loan         563
Vehicle Loan      514
Car Loan          493
car loan          457
Auto Loan         455
CAR LOAN          450
SME Loan          373
NaN               351
business loan     345
Business Loan     340
BUSINESS LOAN     329
Biz Loan          329
Education Loan    241
Student Loan      237
Edu Loan          236
EDUCATION LOAN    230
education loan    229
Unknown            68
                   62
Name: count, dtype: int64

In [59]:
# Normalize

loans['Loan_Type'] = loans['Loan_Type'].str.lower().str.strip()

# Maping into meaningful groups

loan_map = {
    # Personal
    'personal': 'personal',
    'pl': 'personal',
    'personal loan': 'personal',

    # Home
    'mortgage': 'home',
    'home loan': 'home',
    'housing loan': 'home',

    # Vehicle
    'vehicle loan': 'vehicle',
    'car loan': 'vehicle',
    'auto loan': 'vehicle',

    # Business
    'sme loan': 'business',
    'business loan': 'business',
    'biz loan': 'business',

    # Education
    'education loan': 'education',
    'student loan': 'education',
    'edu loan': 'education',

    # Unknown
    'unknown': 'unknown',
    '': 'unknown'
}

loans['Loan_Type'] = loans['Loan_Type'].map(loan_map)


# Handle NaN


loans['Loan_Type'] = loans['Loan_Type'].fillna('unknown')

In [61]:
loans['Loan_Status'].value_counts(dropna=False)

Loan_Status
Open           799
Current        782
Ongoing        779
ongoing        779
Active         775
ACTIVE         748
ONGOING        730
active         716
paid           674
Paid           660
CLOSED         657
Fully Paid     645
Closed         638
Paid Off       637
PAID           628
NaN            360
default        165
Default        159
CHARGED OFF    159
Bad Debt       151
DEFAULT        148
Defaulted      139
Charged Off    127
Unknown         63
                62
Name: count, dtype: int64

In [63]:
# Normalize

loans['Loan_Status'] = loans['Loan_Status'].str.lower().str.strip()

# Maping into meaningful groups

status_map = {
    # Active loans
    'open': 'active',
    'current': 'active',
    'ongoing': 'active',
    'active': 'active',

    # Closed / fully paid
    'paid': 'closed',
    'fully paid': 'closed',
    'paid off': 'closed',
    'closed': 'closed',

    # Defaults / bad loans
    'default': 'default',
    'defaulted': 'default',
    'charged off': 'default',
    'bad debt': 'default',

    # Unknown / blanks
    'unknown': 'unknown',
    '': 'unknown'
}

loans['Loan_Status'] = loans['Loan_Status'].map(status_map)


# Handle NaN

loans['Loan_Status'] = loans['Loan_Status'].fillna('unknown')

In [65]:
print(loans.columns)

Index(['Loan_ID', 'Customer_ID', 'Loan_Amount', 'Loan_Type', 'Interest_Rate',
       'Loan_Term', 'Loan_Status', 'Debt_to_Income_Ratio'],
      dtype='object')


In [67]:
# Handle Duplicates

for col in loans.columns:
    print(col, loans['Loan_ID'].duplicated().sum())

Loan_ID 180
Customer_ID 180
Loan_Amount 180
Loan_Type 180
Interest_Rate 180
Loan_Term 180
Loan_Status 180
Debt_to_Income_Ratio 180


In [69]:
loans.duplicated().sum()

180

In [71]:
# Droping Duplicates

laons = loans.drop_duplicates()

In [73]:
# Check Outlier

num_cols = ['Loan_Amount', 'Interest_Rate', 'Loan_Term', 'Debt_to_Income_Ratio']

for col in num_cols:
    Q1 = loans[col].quantile(0.25)
    Q3 = loans[col].quantile(0.75)
    IQR = Q3 - Q1

    outliers = loans[(loans[col] < Q1 - 1.5*IQR) | 
                        (loans[col] > Q3 + 1.5*IQR)]

    print(f"{col}: {outliers.shape[0]} outliers")

Loan_Amount: 812 outliers
Interest_Rate: 385 outliers
Loan_Term: 52 outliers
Debt_to_Income_Ratio: 1842 outliers


In [75]:
loans['Interest_Rate'].describe()

count    11563.000000
mean        11.938065
std          8.986594
min         -5.000000
25%          9.430000
50%         11.070000
75%         12.990000
max        120.000000
Name: Interest_Rate, dtype: float64

In [77]:
loans = loans[(loans['Interest_Rate'] >= 0) & 
                    (loans['Interest_Rate'] <= 30)]

In [79]:
loans['Interest_Rate'].describe()

count    11387.000000
mean        11.251852
std          2.490027
min          0.000000
25%          9.450000
50%         11.060000
75%         12.960000
max         21.990000
Name: Interest_Rate, dtype: float64

In [81]:
loans['Loan_Term'].value_counts().sort_index()

Loan_Term
-12.0       65
 0.0        72
 12.0     1145
 24.0     1255
 36.0     1133
 48.0     1199
 60.0     1211
 84.0     1200
 120.0    1187
 180.0    1228
 240.0    1167
 600.0      50
Name: count, dtype: int64

In [83]:
# Fix Outlier In Loan_Term

loans = loans[
    (loans['Loan_Term'] > 0) & 
    (loans['Loan_Term'] <= 360)
]


# Creating Buckets

loans['Loan_Term_Category'] = pd.cut(
    loans['Loan_Term'],
    bins=[0, 36, 120, 360],
    labels=['short', 'medium', 'long']
)

In [85]:
loans.describe()

,Loan_Amount,Interest_Rate,Loan_Term,Debt_to_Income_Ratio
count,9.468000e+03,10725.000000,10725.000000,10071.000000
mean,8.385057e+06,11.252065,89.437762,0.795495
std,8.435344e+07,2.483150,72.426720,8.223308
min,-1.000000e+03,0.000000,12.000000,-0.500000
25%,3.280682e+05,9.460000,36.000000,0.000000
50%,7.028590e+05,11.060000,60.000000,0.000000
75%,1.729226e+06,12.950000,120.000000,0.010000
max,1.000000e+09,21.990000,240.000000,99.000000


In [87]:
# Setting upper limit for loan
loans = loans[loans['Loan_Amount'] <= 50000000]

In [89]:
# Null Handling

print(loans.isnull().sum())

Loan_ID                   0
Customer_ID             155
Loan_Amount               0
Loan_Type                 0
Interest_Rate             0
Loan_Term                 0
Loan_Status               0
Debt_to_Income_Ratio    574
Loan_Term_Category        0
dtype: int64


In [91]:
# Null Handling

# customer_id
loans = loans.dropna(subset=['Customer_ID'])

# Loan_amount
loans['Loan_Amount'] = loans['Loan_Amount'].fillna(loans['Loan_Amount'].median())

# Debt To Income Ration
loans['Debt_to_Income_Ratio']  = loans['Debt_to_Income_Ratio'].fillna(loans.groupby('Loan_Type')['Debt_to_Income_Ratio'].transform('median'))

In [93]:
loans.shape

(9245, 9)

# Transaction Data Cleaning

In [96]:
# Checking Nulls

transactions.isnull().sum()

Transaction_ID        0
Loan_ID               0
Payment_Date      57364
Payment_Amount        0
Payment_Status      315
dtype: int64

In [98]:
print(transactions.columns)

Index(['Transaction_ID', 'Loan_ID', 'Payment_Date', 'Payment_Amount',
       'Payment_Status'],
      dtype='object')


In [100]:
# Checking The Different Values In The Loan_ID Column

transactions['Loan_ID'].value_counts(dropna=False)

Loan_ID
LOAN009967    16
LOAN008072    15
LOAN011897    15
LOAN000226    15
LOAN009491    15
              ..
LOAN010909     5
LOAN004511     5
LOAN001651     5
LOAN010565     5
LOAN006277     5
Name: count, Length: 12000, dtype: int64

In [102]:
# Handle Nulls In Loan_ID

transactions = transactions.dropna(subset=['Loan_ID'])

In [104]:
# Converting Blank Values Into Null

transactions['Loan_ID'] = transactions['Loan_ID'].replace(r'^\s*$', np.nan, regex=True)

In [106]:
transactions.isna().sum()

Transaction_ID        0
Loan_ID               0
Payment_Date      57364
Payment_Amount        0
Payment_Status      315
dtype: int64

In [108]:
# Drop Null Again
transactions = transactions.dropna(subset=['Loan_ID'])

In [110]:
# Handle Unknown Values
transactions['Loan_ID'] = transactions['Loan_ID'].replace('Unknown', np.nan)

In [112]:
# Drop Null Again
transactions = transactions.dropna(subset=['Loan_ID'])

In [114]:
transactions['Payment_Date'].value_counts(dropna=False)

Payment_Date
NaN           57364
18-02-2023       10
31-12-2021       10
09-07-2023       10
20-10-2021        9
              ...  
24-09-2021        1
24-02-2022        1
05-01-2022        1
29-07-2022        1
18-11-2023        1
Name: count, Length: 1738, dtype: int64

In [116]:
# Inspect raw values

transactions['Payment_Date'].unique()[:20]

array([nan, '22-12-2024', '25-02-2022', '04-09-2022', '26-08-2024',
       '26-04-2020', '16-08-2021', '14-02-2024', '22-05-2020',
       '11-02-2021', '16-08-2020', '24-08-2020', '09-05-2023',
       '24-01-2021', '25-06-2020', '16-08-2024', '26-03-2022',
       '23-11-2022', '11-05-2024', '19-04-2021'], dtype=object)

In [118]:
# mixed format parsing

transactions['Payment_Date'] = pd.to_datetime(
    transactions['Payment_Date'],
    format='mixed',
    errors='coerce',
    dayfirst=True
)

In [120]:
# Cleaning garbage

transactions['Payment_Date'] = transactions['Payment_Date'].replace(
    ['Unknown', 'TBD', 'N/A', '#NA', ''],
    pd.NA
)

In [122]:
# Strip hidden issues

transactions['Payment_Date'] = transactions['Payment_Date'].astype(str).str.strip()

In [124]:
transactions.shape

(63262, 5)

In [126]:
transactions.isna().sum()

Transaction_ID      0
Loan_ID             0
Payment_Date        0
Payment_Amount      0
Payment_Status    315
dtype: int64

In [128]:
transactions['Payment_Status'].value_counts(dropna=False)

Payment_Status
paid              43344
late              11853
missed             5545
no_transaction     1814
unknown             391
NaN                 315
Name: count, dtype: int64

In [130]:
# Standardize Text
transactions['Payment_Status'] = transactions['Payment_Status'].str.strip().str.lower()

In [132]:
# Converting Blanks To Unknown
transactions['Payment_Status'] = transactions['Payment_Status'].replace(['','Unknown'], np.nan)

In [134]:
# Maping To Clean Categories
status_map = {
    'paid': 'Paid',
    'success': 'Paid',
    'completed': 'Paid',
    'cleared': 'Paid',

    'late': 'Late',
    'delay': 'Late',
    'delayed': 'Late',

    'overdue': 'Overdue',

    'missed': 'Missed',
    'skipped': 'Missed',

    'failed': 'Failed',
    'not paid': 'Failed'
}

transactions['Payment_Status'] = transactions['Payment_Status'].map(status_map)

In [136]:
transactions['Payment_Status'] = transactions['Payment_Status'].fillna('Unknown')

In [138]:
transactions.shape

(63262, 5)

In [140]:
customers.info()
loans.info()
transactions.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4866 entries, 0 to 6089
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Customer_ID          4866 non-null   object 
 1   Age                  4866 non-null   float64
 2   Gender               4866 non-null   object 
 3   Income               4866 non-null   float64
 4   Employment_Type      4866 non-null   object 
 5   Years_of_Employment  4866 non-null   float64
 6   Credit_Score         4866 non-null   float64
dtypes: float64(4), object(3)
memory usage: 304.1+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 9245 entries, 0 to 12179
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   Loan_ID               9245 non-null   object  
 1   Customer_ID           9245 non-null   object  
 2   Loan_Amount           9245 non-null   float64 
 3   Loan_Type             9245 n

In [142]:
loans = loans[loans['Loan_Amount'] > 0]

In [144]:
loans = loans[loans['Loan_Amount'] <= 50000000]

In [146]:
loans = loans[loans['Loan_Amount'] <= 50000000]

In [148]:
loans = loans[loans['Interest_Rate'] > 0]

In [150]:
loans = loans[
    (loans['Debt_to_Income_Ratio'] >= 0) &
    (loans['Debt_to_Income_Ratio'] <= 1)
]

In [152]:
loan_customers = pd.merge(
    loans,
    customers,
    on='Customer_ID',
    how='left'
)

final_df = pd.merge(
    loan_customers,
    transactions,
    on='Loan_ID',
    how='left'
)

In [154]:
final_df.head()

,Loan_ID,Customer_ID,Loan_Amount,Loan_Type,Interest_Rate,Loan_Term,Loan_Status,Debt_to_Income_Ratio,Loan_Term_Category,Age,Gender,Income,Employment_Type,Years_of_Employment,Credit_Score,Transaction_ID,Payment_Date,Payment_Amount,Payment_Status
0,LOAN006290,CUST01050,4255207.0,home,7.44,24.0,default,0.01,short,55.0,female,109302.0,salaried,11.169224,544.0,TXN00033894,NaT,7967.27,Paid
1,LOAN006290,CUST01050,4255207.0,home,7.44,24.0,default,0.01,short,55.0,female,109302.0,salaried,11.169224,544.0,TXN00022652,NaT,8016.75,Late
2,LOAN006290,CUST01050,4255207.0,home,7.44,24.0,default,0.01,short,55.0,female,109302.0,salaried,11.169224,544.0,TXN00000393,2023-12-16,40268.94,Paid
3,LOAN006290,CUST01050,4255207.0,home,7.44,24.0,default,0.01,short,55.0,female,109302.0,salaried,11.169224,544.0,TXN00029375,NaT,9729.34,Late
4,LOAN006290,CUST01050,4255207.0,home,7.44,24.0,default,0.01,short,55.0,female,109302.0,salaried,11.169224,544.0,TXN00049585,2020-11-18,30951.67,Paid


In [156]:
final_df.shape

(47806, 19)

In [158]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47806 entries, 0 to 47805
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   Loan_ID               47806 non-null  object  
 1   Customer_ID           47806 non-null  object  
 2   Loan_Amount           47806 non-null  float64 
 3   Loan_Type             47806 non-null  object  
 4   Interest_Rate         47806 non-null  float64 
 5   Loan_Term             47806 non-null  float64 
 6   Loan_Status           47806 non-null  object  
 7   Debt_to_Income_Ratio  47806 non-null  float64 
 8   Loan_Term_Category    47806 non-null  category
 9   Age                   40737 non-null  float64 
 10  Gender                40737 non-null  object  
 11  Income                40737 non-null  float64 
 12  Employment_Type       40737 non-null  object  
 13  Years_of_Employment   40737 non-null  float64 
 14  Credit_Score          40737 non-null  float64 
 15  Tr

# EDA

In [161]:
# RISK CATEGORY

def risk_category(score):
    if score < 580:
        return 'Poor'
    elif score < 670:
        return 'Fair'
    elif score < 740:
        return 'Good'
    else:
        return 'Excellent'

final_df['Risk_Category'] = final_df['Credit_Score'].apply(risk_category)

In [163]:
# INCOME GROUP

def income_group(x):
    if x < 50000:
        return 'Low Income'
    elif x < 1000000:
        return 'Middle Income'
    else:
        return 'High Income'

final_df['Income_Group'] = final_df['Income'].apply(income_group)

In [165]:
# HIGH RISK FLAG

final_df['High_Risk'] = (
    (final_df['Credit_Score'] < 600) &
    (final_df['Debt_to_Income_Ratio'] > 0.4)
).astype(int)

In [167]:
payment_score_map = {
    'paid': 3,
    'late': 2,
    'missed': 0,
    'no_transaction': 0,
    'unknown': 1
}

final_df['Payment_Behaviour_Score'] = (
    final_df['Payment_Status']
    .str.lower()
    .map(payment_score_map)
)

# CUSTOMER SEGMENTATION

In [170]:
# Selecting columns

cluster_data = final_df[[
    'Income',
    'Credit_Score',
    'Loan_Amount',
    'Debt_to_Income_Ratio'
]].dropna()

In [172]:
# Scale th Data

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled_data = scaler.fit_transform(cluster_data)

In [173]:
# Applying KMeans

from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=3,
    random_state=42
)

cluster_data['Cluster'] = kmeans.fit_predict(scaled_data)

In [174]:
# Cluster Back

final_df.loc[cluster_data.index, 'Cluster'] = cluster_data['Cluster']

In [175]:
cluster_data.groupby('Cluster').mean()

,Income,Credit_Score,Loan_Amount,Debt_to_Income_Ratio
Cluster,,,,
0,91756.469019,748.944811,1.202487e+06,0.011483
1,133751.241797,618.718101,1.175906e+06,0.011508
2,83362.278191,709.595326,1.584522e+06,0.932480


In [176]:
# Creating Segment Names

segment_map = {
    0: 'Reliable Customers',
    1: 'Medium Risk Customers',
    2: 'High DTI Risk Customers'
}

final_df['Customer_Segment'] = final_df['Cluster'].map(segment_map)

In [177]:
final_df['Customer_Segment'].value_counts()

Customer_Segment
Reliable Customers         25516
Medium Risk Customers      11931
High DTI Risk Customers     3290
Name: count, dtype: int64

In [178]:
final_df.groupby('Customer_Segment')[[
    'Income',
    'Credit_Score',
    'Loan_Amount',
    'Debt_to_Income_Ratio'
]].mean()

,Income,Credit_Score,Loan_Amount,Debt_to_Income_Ratio
Customer_Segment,,,,
High DTI Risk Customers,83362.278191,709.595326,1.584522e+06,0.932480
Medium Risk Customers,133751.241797,618.718101,1.175906e+06,0.011508
Reliable Customers,91756.469019,748.944811,1.202487e+06,0.011483


In [192]:
# Import Dataset

final_df.to_excel('loan_analysis_final.xlsx', index=False)

In [189]:
final_df.isnull().sum()

Loan_ID                       0
Customer_ID                   0
Loan_Amount                   0
Loan_Type                     0
Interest_Rate                 0
Loan_Term                     0
Loan_Status                   0
Debt_to_Income_Ratio          0
Loan_Term_Category            0
Age                        7069
Gender                     7069
Income                     7069
Employment_Type            7069
Years_of_Employment        7069
Credit_Score               7069
Transaction_ID                0
Payment_Date                  0
Payment_Amount                0
Payment_Status                0
Risk_Category                 0
Income_Group                  0
High_Risk                     0
Payment_Behaviour_Score       0
Cluster                    7069
Customer_Segment           7069
dtype: int64

In [200]:
final_df['Credit_Score'].describe()

count    40737.000000
mean       707.626239
std         78.704089
min        466.000000
25%        653.000000
50%        706.000000
75%        744.000000
max        954.000000
Name: Credit_Score, dtype: float64